In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import UIOrthoLoRAConfig, TaskType, get_peft_model

In [ ]:
def prepare_dataset(tokenizer, max_len=128, task="sst2"):
    ds = load_dataset("glue", task)
    
    def tokenize_function(examples):
        if task in ["sst2", "cola"]:
            # Single sentence tasks
            return tokenizer(
                examples["sentence"],
                truncation=True,
                padding="max_length",
                max_length=max_len
            )
        elif task in ["mrpc", "qnli", "rte", "wnli", "mnli", "qqp", "sts-b"]:
            # Two sentence tasks
            return tokenizer(
                examples["sentence1"],
                examples["sentence2"],
                truncation=True,
                padding="max_length",
                max_length=max_len
            )
    
    ds = ds.map(tokenize_function, batched=True)
    ds = ds.rename_column("label", "labels")
    ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

In [ ]:
base_id = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(base_id, use_fast=True)

base = AutoModelForSequenceClassification.from_pretrained(
    base_id, num_labels=2, device_map="auto"
)

In [ ]:
uiortholora_cfg = UIOrthoLoRAConfig(
    target_modules=["query", "value"],
    uiortholora_alpha=1.0,
    uiortholora_dropout=0.0,
    fan_in_fan_out=False,
    initial_scaler=0.1,
    initial_sigma=0.1,
    num_svalues_to_adapt=128,
    num_svectors_to_adapt=60,
    task_type=TaskType.SEQ_CLS)
model = get_peft_model(base, uiortholora_cfg)

In [ ]:
model.print_trainable_parameters()

In [ ]:
model.base_model.model.roberta.encoder.layer[2].attention.self.query.get_delta_weight("default")

In [ ]:
ortholora_layer = model.base_model.model.roberta.encoder.layer[0]

In [ ]:
model.base_model.model.roberta.encoder.layer[0].attention.self.query.weight

In [ ]:
adapter_name = "default"
U = getattr(ortholora_layer, f"{adapter_name}_U")
V = getattr(ortholora_layer, f"{adapter_name}_V")

In [ ]:
dir(ortholora_layer)

In [ ]:
ortholora_layer.weight.shape

In [ ]:
print(U.shape)
print(V.shape)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "NousResearch/Llama-2-13b-hf"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
instruction = """Instruction 1:
You are a master storyteller. Tell me a vivid, imaginative, and emotionally engaging story that captures the reader’s attention from the very first sentence.
Your story should include a main character with a clear goal, unexpected obstacles, and a powerful resolution.
Incorporate rich sensory descriptions—what things look like, sound like, smell like.
Include dialogue where appropriate to bring characters to life.
The setting can be fantastical, historical, or realistic, but make sure it is fully developed and immersive.
Keep the tone consistent and build suspense throughout.
Make the reader feel deeply connected to the characters and their journey. Begin the story now.

Instruction 2:
You are a skilled narrator. Create a detailed and imaginative story that pulls the reader into a fully realized world from the opening sentence.
Your protagonist should have a specific goal and encounter both internal and external challenges before arriving at a meaningful resolution.
Describe the environment in vivid detail—what is seen, heard, touched, and smelled.
Use dialogue to enrich character development and move the story forward.
Choose a setting—fantasy, historical, or realistic—and bring it to life with immersive description.
Maintain a consistent voice and use pacing to build tension.
Make the audience care deeply about what happens next. Now, begin the tale.

Instruction 3:
You are an expert writer. Craft a compelling and emotionally powerful story that starts with a hook and never lets go.
Center the plot around a central character with a defined objective.
Introduce setbacks and twists that challenge their values or force them to grow.
Use sensory-rich language to show the world around them.
Allow dialogue to reveal personality and relationships.
Ground the story in a strong setting—be it magical, ancient, or modern.
Keep your tone steady and draw the reader into the emotional arc.
Your mission is to connect deeply with your reader through the power of story. Begin now.

Instruction 4:
Act as a literary genius and construct a mesmerizing narrative with tension, heart, and wonder.
Start your story with a striking first line. Introduce a central character who seeks something important.
Let their path be blocked by unexpected events and difficult decisions.
Paint every scene with words: colors, sounds, feelings, movements, smells.
Use conversations to make relationships real.
The world you build can span galaxies or be a single village—but make it rich and tangible.
Stay true to the tone and pacing.
End with a resolution that leaves a lasting impact. Begin your work of art.

Instruction 5:
You are a celebrated storyteller. Weave an enchanting tale that transports readers to another world or into someone else’s shoes.
Create a protagonist with a purpose—maybe revenge, discovery, redemption, or love.
Make them struggle, grow, and change.
Immerse the reader in the details of the world—rain on rooftops, the scent of pine, whispers in the dark.
Let characters speak their truths through realistic dialogue.
Whether fantasy, dystopia, or drama, make the setting unforgettable.
Keep readers on edge or in awe through careful pacing and tone.
Tell a tale that will be remembered. Begin now.

Instruction 6:
Imagine you are the author of a best-selling novel. Write a story that explores human emotion, conflict, and triumph.
Your main character should face real challenges that test their courage or beliefs.
Use rich, sensory details to help the reader feel immersed.
Include realistic and meaningful dialogue.
Let your setting breathe—it should feel alive whether it’s a city, forest, or spaceship.
Maintain a strong and consistent tone.
Draw your reader into a story that feels both grand and intimate.
Start with a powerful first sentence. Begin writing.

Instruction 7:
Write as if your story will be read by millions. Start with an attention-grabbing hook.
Introduce a relatable or intriguing main character with a clear desire or mission.
Complicate their path with conflict, mystery, or moral dilemma.
Layer the scenes with imagery—what does the character see, hear, taste, smell?
Use natural, purposeful dialogue to advance plot and character arcs.
Let your setting shine—it should support the mood and theme of the story.
Keep readers engaged with pacing and emotional stakes.
This is your masterpiece. Start writing now.

Instruction 8:
Compose a beautifully structured story with emotional depth, exciting twists, and memorable characters.
Make your main character’s journey matter. Show their failures and victories.
Describe every setting with sensory detail so the reader feels they’re truly there.
Let characters speak like real people.
Pick a genre—myth, sci-fi, realism—and commit to it with care.
Pace the story for suspense, surprise, and reflection.
Use your words to inspire empathy, joy, fear, or wonder.
Begin with a bang—and don’t stop until the final, unforgettable line.

"""

In [ ]:
inputs = tokenizer(instruction, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=5)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
import numpy as np
from scipy.linalg import qr

# Step 1: Create a 7x7 orthogonal matrix U via QR decomposition
random_matrix = np.random.randn(7, 7)
U, _ = np.linalg.qr(random_matrix)  # U is orthogonal

# Step 2: Create a 4x4 diagonal matrix (scaling)
D = np.diag([1.0, 1.0, 1.0, 1.0])  # example scaling

# Step 3: Create a 3x3 orthogonal matrix
random_block = np.random.randn(3, 3)
Q3, _ = np.linalg.qr(random_block)

# Step 4: Construct block matrix M
M = np.block([
    [D,             np.zeros((4, 3))],
    [np.zeros((3, 4)), Q3           ]
])

# Apply the transformation from right
U_mod = U @ M

# Apply the transformation from left
U_mod_left = M @ U

# Print to verify
np.set_printoptions(precision=3, suppress=True)
print("Original U:\n", U)
print("\nBlock matrix M:\n", M)
print("\nModified U (U @ M):\n", U_mod)
print("\nModified U (M @ U):\n", U_mod_left)


In [ ]:
import numpy as np

np.random.seed(0)
np.set_printoptions(precision=3, suppress=True)

# Random dense 5x5 matrix
A = np.random.randn(5, 5)

# Diagonal 5x5 with 3 nonzero entries; bottom-right 2 entries are zero
d = np.array([np.random.randn(), np.random.randn(), np.random.randn(), 0.0, 0.0])
D = np.diag(d)

print("Dense matrix A:")
print(A)
print("\nDiagonal matrix D (last two diagonal entries = 0):")
print(D)

print("\nA @ D  (D from the RIGHT  -> zeros out the last 2 COLUMNS of A):")
print(A @ D)

print("\nD @ A  (D from the LEFT   -> zeros out the last 2 ROWS of A):")
print(D @ A)
